# Controllers — sampling-MPC algorithms

This is the home base for the controllers. **The algorithm code lives in the
package** (`analytic_mppi/controllers/`), not in these cells, so every
environment notebook imports the *same* latest code. With `%autoreload 2` on,
you edit a `.py` and re-run — no kernel restart.

- Baselines: `analytic_mppi/controllers/` — `mppi`, `mppi_cma`, `cem`, `dial`, `predictive_sampling`
- Experimental: `analytic_mppi/controllers/experimental.py` — `uniform_gd`, `gaussian_gd`, `rank_cma`, `fpl_value_cma`

Each section below briefly describes one algorithm and runs a tiny pendulum demo.
Tweak a class in its `.py`, re-run that section, see the effect immediately.

In [ ]:
# Always pick up the latest code in analytic_mppi/ (incl. experimental.py)
# without restarting the kernel — edit a .py, re-run a cell, done.
%load_ext autoreload
%autoreload 2

import numpy as np
from analytic_mppi.tasks import make_task
from analytic_mppi.eval import make_controller, ALL_CONTROLLERS

print("registered controllers:", sorted(ALL_CONTROLLERS))

def demo(controller, *, cost_mode="normal", steps=5, **kw):
    """Build `controller` on the pendulum, run a few closed-loop steps, summarise."""
    t, b, c = make_controller("pendulum", controller, cost_mode=cost_mode,
                              num_samples=32, plan_horizon=0.5, num_knots=4, **kw)
    s = b.get_state()
    for _ in range(steps):
        s = b.step(c.act(s))
    assert np.all(np.isfinite(s))
    print(f"{type(c).__name__:14s} ok  |  K={c.num_samples} num_knots={c.num_knots} "
          f"H={c.H} cost={cost_mode}")
    return c

## MPPI  (`mppi`)

Knot-spline Gaussian sampling. Normal cost: softmax-weighted average of the sampled knots. FPL: best-sample selection. The vanilla baseline.

In [ ]:
demo("mppi", noise_level=0.5, temperature=1.0)

## MPPI-CMA  (`mppi_cma`)

MPPI with per-knot covariance adaptation (block-diagonal CMA). Mean is the softmax-weighted average; covariance is an EMA of the weighted outer products, eigenvalue-floored at `minimum_noise_level**2`.

In [ ]:
demo("mppi_cma", initial_noise_level=0.5, temperature=1.0, minimum_noise_level=0.1)

## CEM  (`cem`)

Cross-Entropy Method: each step, fit a diagonal Gaussian to the top-`num_elites` rollouts and clamp sigma to `sigma_min`.

In [ ]:
demo("cem", num_elites=8, sigma_start=1.0, sigma_min=0.1)

## DIAL  (`dial`)

Diffusion-Inspired Annealing: per-knot, per-iteration noise schedule that anneals across optimization iterations (`beta_opt_iter`) and along the horizon (`beta_horizon`).

In [ ]:
demo("dial", noise_level=0.5, temperature=1.0, beta_opt_iter=3.0, beta_horizon=3.0)

## Predictive Sampling  (`predictive_sampling`)

Greedy best-of-K. The current mean is always included as a sample, so the controller can never regress.

In [ ]:
demo("predictive_sampling", noise_level=0.5)

## UniformGD  (`uniform_gd`)

Knot-free (one knot per step). Per-step **Uniform(-w, +w)** noise around the previous-best rollout, smoothed by a few curvature-GD steps, best-of-K update. Built for the FPL fulfillment path.

In [ ]:
demo("uniform_gd", noise_level=0.5, n_gd_iter=3)

## GaussianGD  (`gaussian_gd`)

Same per-step, best-of-K, curvature-smoothed recipe as UniformGD, but with **Gaussian** proposal noise instead of uniform.

In [ ]:
demo("gaussian_gd", noise_level=0.3, n_gd_iter=3)

## RankCMA  (`rank_cma`)

Per-step multivariate Gaussian with a within-step **rank-weighted CMA** refit (log-rank positives + an active-CMA negative tail). The per-step covariance warm-shifts across MPC steps so elite directions accumulate. `use_cov_update=True` turns on the covariance adaptation (otherwise cov stays isotropic).

In [ ]:
demo("rank_cma", sigma_init=0.5, n_gd_iter=3)

## FplValueCMA  (`fpl_value_cma`)

RankCMA's log-rank prior **times** an FPL value-softmax bonus on the elite cohort. `beta` sets how strongly absolute FPL reward dominates the rank prior. **Requires FPL mode** — normal cost has no fixed reward scale.

In [ ]:
demo("fpl_value_cma", cost_mode="fpl_discounted", sigma_init=0.5, beta=10.0, n_gd_iter=3)

## Cost-GD refinement  (`cost_gd=...`)

A refinement **wrapper** for any controller above, not a standalone sampler. Each MPC step
it takes a few gradient-descent steps on the trajectory cost of the top-mu rollouts — the
gradient comes from reverse-mode BPTT through per-step `mujoco.mjd_transitionFD` Jacobians,
with a closed-form cost-aggregation gradient (verified against finite differences to ~1e-7).

Turn it on by passing `cost_gd=dict(gd_iterations=N, gd_lr=...)` to `make_controller`, or
`Config(..., cost_gd=dict(...))` in an env notebook.

- **Requires `nq == nv`** (pendulum / walker / hopper); raises on cube / g1_standup (free-joint quaternions).
- `gd_iterations=0` reproduces the base controller bit-for-bit.

In [ ]:
from analytic_mppi.controllers.cost_gd import wrap_controller_with_gd_refine

# Any base controller + cost_gd=... ; here RankCMA on the pendulum under FPL.
t, b, c = make_controller("pendulum", "rank_cma", cost_mode="fpl_discounted",
                          num_samples=32, plan_horizon=0.5, num_knots=4,
                          sigma_init=0.5, n_gd_iter=3,
                          cost_gd=dict(gd_iterations=3, gd_lr=0.1))
s = b.get_state()
for _ in range(5):
    s = b.step(c.act(s))
print("cost-GD wrapped:", c._costgd_wrapped, " num_refine:", c._costgd_num_refine,
      " gd_iters:", c._costgd_iterations)